In [1]:
import pandas as pd
import numpy as np
import boto3
from io import BytesIO
from scipy import stats
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

s3 = boto3.client(
    "s3",
    endpoint_url="http://minio:9000",
    aws_access_key_id="minioadmin",
    aws_secret_access_key="minioadmin123",
)
BUCKET = "oil-pipeline"

def read_parquet(key):
    obj = s3.get_object(Bucket=BUCKET, Key=key)
    return pd.read_parquet(BytesIO(obj["Body"].read()))

def save_parquet(df, key):
    buf = BytesIO()
    df.to_parquet(buf, index=False)
    buf.seek(0)
    s3.put_object(Bucket=BUCKET, Key=key, Body=buf.getvalue())
    print(f"Saved: {key} ({len(df)} rows)")

In [2]:
sensors = read_parquet("cleaned/pump_sensors/data.parquet")
failures = read_parquet("cleaned/pumps/pumps.parquet")
pumps = read_parquet("cleaned/pumps/pumps.parquet")

sensors["timestamp"] = pd.to_datetime(sensors["timestamp"])
sensors = sensors.sort_values(["pump_id", "timestamp"])

print(sensors.shape)
sensors.head()

(72, 8)


,record_id,pump_id,timestamp,temperature,vibration,current,rpm,pressure
0,1,1,2025-10-01 00:00:00,72.3,2.1,58.2,1470.0,122.4
1,2,1,2025-10-01 03:00:00,72.6,2.0,58.4,1472.0,122.5
2,3,1,2025-10-01 06:00:00,73.1,2.2,58.6,1474.0,122.6
3,4,1,2025-10-01 09:00:00,72.8,2.1,58.3,1471.0,122.3
4,5,1,2025-10-01 12:00:00,73.0,2.3,58.5,1473.0,122.7


In [3]:
features = ["temperature", "vibration", "current", "rpm", "pressure"]

# считаем z-score по каждому насосу отдельно
sensors["anomaly_zscore"] = 0

for pump_id in sensors["pump_id"].unique():
    mask = sensors["pump_id"] == pump_id
    subset = sensors.loc[mask, features]
    z = np.abs(stats.zscore(subset))
    # если хотя бы один показатель > 2 - аномалия
    sensors.loc[mask, "anomaly_zscore"] = (z > 2).any(axis=1).astype(int)

print("Аномалии по z-score:")
print(sensors.groupby("pump_id")["anomaly_zscore"].sum())

Аномалии по z-score:
pump_id
1    2
3    2
5    1
Name: anomaly_zscore, dtype: int64


In [4]:
scaler = StandardScaler()
X = scaler.fit_transform(sensors[features])

iso = IsolationForest(contamination=0.15, random_state=42)
sensors["anomaly_iso"] = (iso.fit_predict(X) == -1).astype(int)

print("Аномалии по Isolation Forest:")
print(sensors.groupby("pump_id")["anomaly_iso"].sum())

Аномалии по Isolation Forest:
pump_id
1    2
3    2
5    7
Name: anomaly_iso, dtype: int64


In [5]:
# отказы произошли 2025-10-04, смотрим тренд за 3 дня до
print("Тренд показателей по дням перед отказом:\n")

sensors["date"] = sensors["timestamp"].dt.date

trend = sensors.groupby(["pump_id", "date"]).agg(
    avg_temp=("temperature", "mean"),
    avg_vibration=("vibration", "mean"),
    avg_current=("current", "mean"),
).reset_index()

print(trend.to_string(index=False))

Тренд показателей по дням перед отказом:

 pump_id       date  avg_temp  avg_vibration  avg_current
       1 2025-10-01   73.0750         2.3000      58.6500
       1 2025-10-02   74.2000         2.7500      59.2500
       1 2025-10-03   81.1000         6.2000      62.8375
       3 2025-10-01   69.6500         1.9375      54.3875
       3 2025-10-02   71.3125         2.9250      55.3625
       3 2025-10-03   77.1500         6.1750      58.1875
       5 2025-10-01   71.7000         2.7875      56.8750
       5 2025-10-02   76.0750         6.4250      60.0500
       5 2025-10-03   84.9750        15.2125      63.9500


In [6]:
# нормализуем показатели и считаем risk score
latest = sensors.groupby("pump_id").last().reset_index()

for col in ["temperature", "vibration", "current"]:
    col_min = sensors[col].min()
    col_max = sensors[col].max()
    latest[f"{col}_norm"] = (latest[col] - col_min) / (col_max - col_min)

latest["risk_score"] = (
    latest["temperature_norm"] * 0.4 +
    latest["vibration_norm"] * 0.4 +
    latest["current_norm"] * 0.2
).round(3)

risk = latest[["pump_id", "temperature", "vibration", "current", "risk_score"]]
risk = risk.merge(pumps[["pump_id", "well_id", "type"]], on="pump_id")
risk = risk.sort_values("risk_score", ascending=False)

print(risk)
save_parquet(risk, "marts/mart_pump_risk.parquet")

   pump_id  temperature  vibration  current  risk_score  well_id  \
2        5         89.3       20.5     65.5       1.000        5   
0        1         85.1        9.1     65.2       0.669        1   
1        3         81.4        9.4     60.0       0.511        2   

              type  
2          ESP-450  
0          ESP-500  
1  Centrifugal-300  
Saved: marts/mart_pump_risk.parquet (3 rows)


In [7]:
save_parquet(
    sensors[["record_id", "pump_id", "timestamp", "temperature",
             "vibration", "current", "anomaly_zscore", "anomaly_iso"]],
    "marts/mart_pump_anomalies.parquet"
)
print("Done!")

Saved: marts/mart_pump_anomalies.parquet (72 rows)
Done!
